# 07 — Selección auditable de características para nivel de riesgo

Este notebook construye un **ranking de evidencia**, no una búsqueda automática de la combinación que mejor rinda en 2025. Todas las estimaciones usan exclusivamente los registros de entrenamiento 2020–2023. La validación 2024 y la prueba 2025 quedan fuera de esta etapa.

Se combinan: (1) elegibilidad conceptual y temporal definida en los notebooks 03–05; (2) asociación bivariada mediante chi-cuadrado y V de Cramér; (3) información mutua; (4) `feature_importances_` de Extra Trees; (5) importancia por permutación del mismo modelo evaluada dentro de 2020–2023; y (6) RFECV con regresión logística y codificación one-hot, como análisis de sensibilidad comparable con el repositorio previo. El consenso se usa para priorizar, no para afirmar causalidad.

Sustento: Guyon y Elisseeff (2003), [*An Introduction to Variable and Feature Selection*](https://jmlr.org/papers/v3/guyon03a.html), describen la selección como apoyo al desempeño, costo e interpretación. La separación temporal se mantiene según el protocolo registrado en `registro_evidencia_metodologica.md`.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import RFECV, mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'salidas_modelado' / 'bases_finales_riesgo'
OUTPUT_DIR = BASE_DIR / 'salidas_modelado' / 'seleccion_caracteristicas_riesgo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'NIVEL_DE_RIESGO_VICTIMA'
RANDOM_SEED = 20260827
MAX_FILAS_MI = 200_000
MAX_FILAS_ARBOLES = 200_000
MAX_FILAS_PERMUTACION = 50_000
MAX_FILAS_RFECV = 100_000
N_ARBOLES = 250
REPETICIONES_PERMUTACION = 3
EJECUTAR_RFECV = True  # Sensibilidad comparable con el repositorio previo; tarda más.
EJECUTAR_SELECCION = True  # Cambiar a True para ejecutar.

print('Salida:', OUTPUT_DIR)

Salida: c:\Users\jcollazos\CODEX\salidas_modelado\seleccion_caracteristicas_riesgo


In [2]:
def cargar_train(escenario):
    ruta = DATA_DIR / f'riesgo_{escenario}_train_2020_2023.parquet'
    tabla = pd.read_parquet(ruta)
    assert tabla.isna().sum().sum() == 0, f'Hay nulos en {ruta.name}'
    return tabla

def definir_features(tabla):
    excluir = {TARGET, 'nivel_riesgo_etiqueta', 'anio_ingreso'}
    features = [col for col in tabla.columns if col not in excluir]
    numericas = ['EDAD_VICTIMA']
    categoricas = [col for col in features if col not in numericas]
    return features, numericas, categoricas

def muestra_estratificada(tabla, max_filas):
    if len(tabla) <= max_filas:
        return tabla.copy()
    muestra, _ = train_test_split(
        tabla, train_size=max_filas, stratify=tabla[TARGET], random_state=RANDOM_SEED
    )
    return muestra.copy()

def cramers_v(tabla, feature):
    serie = tabla[feature]
    if feature == 'EDAD_VICTIMA':
        # Solo para la comparación bivariada: deciles con tamaño similar.
        serie = pd.qcut(pd.to_numeric(serie, errors='raise'), q=10, duplicates='drop').astype('string')
    contingencia = pd.crosstab(serie, tabla[TARGET])
    chi2, p_valor, _, _ = chi2_contingency(contingencia, correction=False)
    n = contingencia.to_numpy().sum()
    minimo = min(contingencia.shape) - 1
    v = np.sqrt(chi2 / (n * minimo)) if n and minimo else np.nan
    return chi2, p_valor, v

def preparar_codificado(tabla, features, categoricas, encoder=None, ajustar=True):
    resultado = tabla[features].copy()
    resultado['EDAD_VICTIMA'] = pd.to_numeric(resultado['EDAD_VICTIMA'], errors='raise')
    if encoder is None:
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    if categoricas:
        if ajustar:
            resultado[categoricas] = encoder.fit_transform(resultado[categoricas])
        else:
            resultado[categoricas] = encoder.transform(resultado[categoricas])
    return resultado, encoder

for escenario in ['inicial', 'retrospectiva']:
    tabla = cargar_train(escenario)
    features, _, _ = definir_features(tabla)
    print(f'{escenario}: {len(tabla):,} filas, {len(features)} características')

inicial: 598,807 filas, 17 características
retrospectiva: 598,807 filas, 18 características


## Regla de lectura del consenso

Con muestras grandes, un valor p pequeño puede aparecer incluso con asociaciones triviales; por eso no se aprueba una variable solo por significancia. `V de Cramér`, información mutua, `feature_importances_` e importancia por permutación se reportan juntos. La importancia por impureza se conserva para comparar con el repositorio previo, pero puede favorecer variables con muchas categorías; por eso la importancia por permutación recibe más peso interpretativo. RFECV se reporta como sensibilidad: selecciona columnas one-hot, de modo que puede retener solo algunos niveles de una variable y no define por sí mismo la exclusión de la variable original. Una variable es **prioritaria** si queda en la mitad superior del ranking en al menos dos de las tres evidencias principales (V, información mutua y permutación); **exploratoria** si ocurre en una; y de **bajo consenso** si no ocurre en ninguna. La decisión final en el notebook 08 comparará el conjunto completo con el priorizado, usando solo 2024.

In [3]:
def seleccionar_features(escenario):
    tabla = cargar_train(escenario)
    features, _, categoricas = definir_features(tabla)

    # Asociación bivariada en todo el periodo de entrenamiento, sin datos futuros.
    bivariado = []
    for feature in features:
        chi2, p_valor, v = cramers_v(tabla, feature)
        bivariado.append({'feature': feature, 'chi2': chi2, 'p_valor_chi2': p_valor, 'cramers_v': v})
    ranking = pd.DataFrame(bivariado)

    # Información mutua: muestra estratificada para que la ejecución sea reproducible y manejable.
    muestra_mi = muestra_estratificada(tabla, MAX_FILAS_MI)
    X_mi, encoder_mi = preparar_codificado(muestra_mi, features, categoricas)
    discretas = [feature in categoricas for feature in features]
    mi = mutual_info_classif(X_mi, muestra_mi[TARGET], discrete_features=discretas, random_state=RANDOM_SEED)
    ranking = ranking.merge(pd.DataFrame({'feature': features, 'informacion_mutua': mi}), on='feature')

    # Importancia por permutación: modelo y conjunto de evaluación separados dentro de 2020–2023.
    muestra_arbol = muestra_estratificada(tabla, MAX_FILAS_ARBOLES)
    ajuste, seleccion_interna = train_test_split(
        muestra_arbol, test_size=0.25, stratify=muestra_arbol[TARGET], random_state=RANDOM_SEED
    )
    if len(seleccion_interna) > MAX_FILAS_PERMUTACION:
        seleccion_interna = muestra_estratificada(seleccion_interna, MAX_FILAS_PERMUTACION)
    X_ajuste, encoder = preparar_codificado(ajuste, features, categoricas)
    X_seleccion, _ = preparar_codificado(seleccion_interna, features, categoricas, encoder=encoder, ajustar=False)
    arboles = ExtraTreesClassifier(
        n_estimators=N_ARBOLES, class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED, max_features='sqrt'
    )
    arboles.fit(X_ajuste, ajuste[TARGET])
    perm = permutation_importance(
        arboles, X_seleccion, seleccion_interna[TARGET], scoring='balanced_accuracy',
        n_repeats=REPETICIONES_PERMUTACION, n_jobs=-1, random_state=RANDOM_SEED
    )
    ranking = ranking.merge(pd.DataFrame({
        'feature': features,
        'importancia_arboles_impureza': arboles.feature_importances_,
        'importancia_permutacion_media': perm.importances_mean,
        'importancia_permutacion_desv': perm.importances_std,
    }), on='feature')

    # RFECV es una sensibilidad adicional. Opera sobre columnas one-hot; luego se resume por variable original.
    if EJECUTAR_RFECV:
        muestra_rfe = muestra_estratificada(tabla, MAX_FILAS_RFECV)
        muestra_rfe['EDAD_VICTIMA'] = pd.to_numeric(muestra_rfe['EDAD_VICTIMA'], errors='raise')
        preprocesador_rfe = ColumnTransformer([
            ('edad', StandardScaler(), ['EDAD_VICTIMA']),
            ('categorias', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categoricas),
        ])
        X_rfe = preprocesador_rfe.fit_transform(muestra_rfe[features])
        selector_rfe = RFECV(
            estimator=LogisticRegression(max_iter=1000, solver='saga', class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED),
            step=0.25, min_features_to_select=10,
            cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED),
            scoring='balanced_accuracy', n_jobs=-1
        )
        selector_rfe.fit(X_rfe, muestra_rfe[TARGET])
        nombres_one_hot = preprocesador_rfe.get_feature_names_out()
        mapa_origen = []
        for nombre in nombres_one_hot:
            if nombre.startswith('edad__'):
                mapa_origen.append('EDAD_VICTIMA')
            else:
                resto = nombre.replace('categorias__', '', 1)
                mapa_origen.append(next(col for col in categoricas if resto.startswith(f'{col}_')))
        detalle_rfe = pd.DataFrame({'feature': mapa_origen, 'seleccionada_rfe': selector_rfe.support_})
        resumen_rfe = detalle_rfe.groupby('feature', as_index=False).agg(
            componentes_rfe_seleccionados=('seleccionada_rfe', 'sum'),
            componentes_rfe_total=('seleccionada_rfe', 'size')
        )
        resumen_rfe['proporcion_componentes_rfe'] = resumen_rfe['componentes_rfe_seleccionados'] / resumen_rfe['componentes_rfe_total']
        ranking = ranking.merge(resumen_rfe, on='feature', how='left')
    else:
        ranking['componentes_rfe_seleccionados'] = np.nan
        ranking['componentes_rfe_total'] = np.nan
        ranking['proporcion_componentes_rfe'] = np.nan

    criterios = ['cramers_v', 'informacion_mutua', 'importancia_permutacion_media']
    for criterio in criterios:
        ranking[f'rango_{criterio}'] = ranking[criterio].rank(method='min', ascending=False).astype(int)
    limite_superior = math.ceil(len(features) / 2)
    columnas_rango = [f'rango_{criterio}' for criterio in criterios]
    ranking['metodos_mitad_superior'] = (ranking[columnas_rango] <= limite_superior).sum(axis=1)
    ranking['promedio_rango'] = ranking[columnas_rango].mean(axis=1)
    ranking['estado_propuesto'] = np.select(
        [ranking['metodos_mitad_superior'] >= 2, ranking['metodos_mitad_superior'] == 1],
        ['prioritaria', 'exploratoria'], default='bajo_consenso'
    )
    ranking['escenario'] = escenario
    ranking = ranking.sort_values(['metodos_mitad_superior', 'promedio_rango'], ascending=[False, True]).reset_index(drop=True)
    ranking.insert(0, 'orden_consenso', ranking.index + 1)
    columnas = ['orden_consenso', 'escenario', 'feature', 'estado_propuesto', 'metodos_mitad_superior',
                'cramers_v', 'p_valor_chi2', 'informacion_mutua', 'importancia_arboles_impureza',
                'importancia_permutacion_media', 'importancia_permutacion_desv',
                'componentes_rfe_seleccionados', 'componentes_rfe_total', 'proporcion_componentes_rfe',
                'promedio_rango'] + columnas_rango
    ranking = ranking[columnas]
    ruta = OUTPUT_DIR / f'ranking_features_{escenario}_train_2020_2023.csv'
    ranking.to_csv(ruta, index=False, encoding='utf-8-sig')
    return ranking

if EJECUTAR_SELECCION:
    resultados = {escenario: seleccionar_features(escenario) for escenario in ['inicial', 'retrospectiva']}
    for escenario, ranking in resultados.items():
        print(f'\n{escenario.upper()} — ranking de consenso')
        display(ranking)
else:
    print('Selección no ejecutada. Cambie EJECUTAR_SELECCION a True y ejecute esta celda.')

C:\Users\jcollazos\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
C:\Users\jcollazos\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
C:\Users\jcollazos\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs


INICIAL — ranking de consenso


,orden_consenso,escenario,feature,estado_propuesto,metodos_mitad_superior,cramers_v,p_valor_chi2,informacion_mutua,importancia_arboles_impureza,importancia_permutacion_media,importancia_permutacion_desv,componentes_rfe_seleccionados,componentes_rfe_total,proporcion_componentes_rfe,promedio_rango,rango_cramers_v,rango_informacion_mutua,rango_importancia_permutacion_media
0,1,inicial,VINCULO_AGRESOR_VICTIMA,prioritaria,3,0.080349,0.000000e+00,0.006473,0.034444,0.018920,0.000459,3,3,1.0,2.333333,4,1,2
1,2,inicial,PRIMERA_VEZ_AGREDE,prioritaria,3,0.108435,0.000000e+00,0.005740,0.022283,0.012576,0.000341,2,2,1.0,3.000000,1,3,5
2,3,inicial,ESTADO_AGRESOR_U_A,prioritaria,3,0.082492,0.000000e+00,0.006353,0.036888,0.011557,0.000887,4,4,1.0,3.333333,2,2,6
3,4,inicial,EDAD_VICTIMA,prioritaria,3,0.060384,0.000000e+00,0.005212,0.542917,0.025790,0.001144,1,1,1.0,4.333333,8,4,1
4,5,inicial,NIVEL_EDUCATIVO_VICTIMA,prioritaria,3,0.064958,0.000000e+00,0.004499,0.154597,0.018194,0.002361,12,12,1.0,5.000000,7,5,3
5,6,inicial,AREA_RESIDENCIA_DOMICILIO,prioritaria,3,0.037387,0.000000e+00,0.001601,0.043566,0.010120,0.001333,4,4,1.0,8.666667,9,9,8
6,7,inicial,REDES_FAM_SOC,prioritaria,2,0.080442,0.000000e+00,0.003102,0.015569,0.003904,0.000905,2,2,1.0,7.333333,3,6,13
7,8,inicial,SEXO_AGRESOR,prioritaria,2,0.076188,0.000000e+00,0.002852,0.012965,0.004835,0.001330,2,2,1.0,7.666667,5,7,11
8,9,inicial,SEXO_VICTIMA,prioritaria,2,0.071951,0.000000e+00,0.002613,0.013143,0.006262,0.001348,2,2,1.0,8.000000,6,8,10
9,10,inicial,AGRESOR_VIVE_CASA_VICTIMA,exploratoria,1,0.035223,0.000000e+00,0.001281,0.043028,0.012880,0.002466,4,4,1.0,8.666667,11,11,4



RETROSPECTIVA — ranking de consenso


,orden_consenso,escenario,feature,estado_propuesto,metodos_mitad_superior,cramers_v,p_valor_chi2,informacion_mutua,importancia_arboles_impureza,importancia_permutacion_media,importancia_permutacion_desv,componentes_rfe_seleccionados,componentes_rfe_total,proporcion_componentes_rfe,promedio_rango,rango_cramers_v,rango_informacion_mutua,rango_importancia_permutacion_media
0,1,retrospectiva,TIPO_VIOLENCIA,prioritaria,3,0.190266,0.000000e+00,0.036417,0.106191,0.055319,0.000761,4,4,1.000000,1.000000,1,1,1
1,2,retrospectiva,VINCULO_AGRESOR_VICTIMA,prioritaria,3,0.080349,0.000000e+00,0.006473,0.024658,0.019235,0.001982,2,3,0.666667,3.333333,5,2,3
2,3,retrospectiva,PRIMERA_VEZ_AGREDE,prioritaria,3,0.108435,0.000000e+00,0.005740,0.024488,0.013355,0.000978,2,2,1.000000,3.666667,2,4,5
3,4,retrospectiva,ESTADO_AGRESOR_U_A,prioritaria,3,0.082492,0.000000e+00,0.006353,0.031935,0.010873,0.001013,4,4,1.000000,4.333333,3,3,7
4,5,retrospectiva,EDAD_VICTIMA,prioritaria,3,0.060384,0.000000e+00,0.005212,0.484292,0.025030,0.000735,0,1,0.000000,5.333333,9,5,2
5,6,retrospectiva,NIVEL_EDUCATIVO_VICTIMA,prioritaria,3,0.064958,0.000000e+00,0.004499,0.139723,0.016415,0.001569,11,12,0.916667,6.000000,8,6,4
6,7,retrospectiva,REDES_FAM_SOC,prioritaria,2,0.080442,0.000000e+00,0.003102,0.012210,0.004274,0.000495,2,2,1.000000,7.666667,4,7,12
7,8,retrospectiva,SEXO_AGRESOR,prioritaria,2,0.076188,0.000000e+00,0.002852,0.010210,0.001854,0.001104,1,2,0.500000,9.333333,6,8,14
8,9,retrospectiva,SEXO_VICTIMA,prioritaria,2,0.071951,0.000000e+00,0.002613,0.011087,0.002303,0.000792,0,2,0.000000,9.666667,7,9,13
9,10,retrospectiva,AGRESOR_VIVE_CASA_VICTIMA,exploratoria,1,0.035223,0.000000e+00,0.001281,0.040572,0.009523,0.002136,3,4,0.750000,10.666667,12,12,8


In [4]:
# Lectura de resultados ya guardados, sin recalcular.
for escenario in ['inicial', 'retrospectiva']:
    ruta = OUTPUT_DIR / f'ranking_features_{escenario}_train_2020_2023.csv'
    if ruta.exists():
        ranking = pd.read_csv(ruta)
        print(f'\n{escenario.upper()} — variables prioritarias propuestas')
        display(ranking.query("estado_propuesto == 'prioritaria'"))
    else:
        print(f'No existe todavía: {ruta.name}')


INICIAL — variables prioritarias propuestas


,orden_consenso,escenario,feature,estado_propuesto,metodos_mitad_superior,cramers_v,p_valor_chi2,informacion_mutua,importancia_arboles_impureza,importancia_permutacion_media,importancia_permutacion_desv,componentes_rfe_seleccionados,componentes_rfe_total,proporcion_componentes_rfe,promedio_rango,rango_cramers_v,rango_informacion_mutua,rango_importancia_permutacion_media
0,1,inicial,VINCULO_AGRESOR_VICTIMA,prioritaria,3,0.080349,0.0,0.006473,0.034444,0.018920,0.000459,3,3,1.0,2.333333,4,1,2
1,2,inicial,PRIMERA_VEZ_AGREDE,prioritaria,3,0.108435,0.0,0.005740,0.022283,0.012576,0.000341,2,2,1.0,3.000000,1,3,5
2,3,inicial,ESTADO_AGRESOR_U_A,prioritaria,3,0.082492,0.0,0.006353,0.036888,0.011557,0.000887,4,4,1.0,3.333333,2,2,6
3,4,inicial,EDAD_VICTIMA,prioritaria,3,0.060384,0.0,0.005212,0.542917,0.025790,0.001144,1,1,1.0,4.333333,8,4,1
4,5,inicial,NIVEL_EDUCATIVO_VICTIMA,prioritaria,3,0.064958,0.0,0.004499,0.154597,0.018194,0.002361,12,12,1.0,5.000000,7,5,3
5,6,inicial,AREA_RESIDENCIA_DOMICILIO,prioritaria,3,0.037387,0.0,0.001601,0.043566,0.010120,0.001333,4,4,1.0,8.666667,9,9,8
6,7,inicial,REDES_FAM_SOC,prioritaria,2,0.080442,0.0,0.003102,0.015569,0.003904,0.000905,2,2,1.0,7.333333,3,6,13
7,8,inicial,SEXO_AGRESOR,prioritaria,2,0.076188,0.0,0.002852,0.012965,0.004835,0.001330,2,2,1.0,7.666667,5,7,11
8,9,inicial,SEXO_VICTIMA,prioritaria,2,0.071951,0.0,0.002613,0.013143,0.006262,0.001348,2,2,1.0,8.000000,6,8,10



RETROSPECTIVA — variables prioritarias propuestas


,orden_consenso,escenario,feature,estado_propuesto,metodos_mitad_superior,cramers_v,p_valor_chi2,informacion_mutua,importancia_arboles_impureza,importancia_permutacion_media,importancia_permutacion_desv,componentes_rfe_seleccionados,componentes_rfe_total,proporcion_componentes_rfe,promedio_rango,rango_cramers_v,rango_informacion_mutua,rango_importancia_permutacion_media
0,1,retrospectiva,TIPO_VIOLENCIA,prioritaria,3,0.190266,0.0,0.036417,0.106191,0.055319,0.000761,4,4,1.000000,1.000000,1,1,1
1,2,retrospectiva,VINCULO_AGRESOR_VICTIMA,prioritaria,3,0.080349,0.0,0.006473,0.024658,0.019235,0.001982,2,3,0.666667,3.333333,5,2,3
2,3,retrospectiva,PRIMERA_VEZ_AGREDE,prioritaria,3,0.108435,0.0,0.005740,0.024488,0.013355,0.000978,2,2,1.000000,3.666667,2,4,5
3,4,retrospectiva,ESTADO_AGRESOR_U_A,prioritaria,3,0.082492,0.0,0.006353,0.031935,0.010873,0.001013,4,4,1.000000,4.333333,3,3,7
4,5,retrospectiva,EDAD_VICTIMA,prioritaria,3,0.060384,0.0,0.005212,0.484292,0.025030,0.000735,0,1,0.000000,5.333333,9,5,2
5,6,retrospectiva,NIVEL_EDUCATIVO_VICTIMA,prioritaria,3,0.064958,0.0,0.004499,0.139723,0.016415,0.001569,11,12,0.916667,6.000000,8,6,4
6,7,retrospectiva,REDES_FAM_SOC,prioritaria,2,0.080442,0.0,0.003102,0.012210,0.004274,0.000495,2,2,1.000000,7.666667,4,7,12
7,8,retrospectiva,SEXO_AGRESOR,prioritaria,2,0.076188,0.0,0.002852,0.010210,0.001854,0.001104,1,2,0.500000,9.333333,6,8,14
8,9,retrospectiva,SEXO_VICTIMA,prioritaria,2,0.071951,0.0,0.002613,0.011087,0.002303,0.000792,0,2,0.000000,9.666667,7,9,13


## Decisión que se documentará antes del notebook 08

No se excluirá una variable únicamente por aparecer como `bajo_consenso`: puede mantenerse en el modelo completo si es conceptualmente necesaria. El notebook 08 contrastará, para cada escenario, un modelo con todas las variables elegibles y otro con las prioritarias. La elección se hará por macro-F1, balanced accuracy, recall y precisión de Severo en 2024.

Antes de continuar, registrar en `registro_evidencia_metodologica.md`: variables priorizadas, variables retenidas por criterio conceptual y cualquier diferencia entre ambos escenarios.